# Experimento

### criando o meta conhecimento
---

Criação do meta dataset

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openml
from pathlib import Path
from time import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import cross_validate, StratifiedKFold, LeaveOneOut
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, f1_score

from pymfe.mfe import MFE

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

c:\Users\Victo\Desktop\Ufal\2026.1\cc-meta-aprendizagem-2026.1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def save_csv(data: pd.DataFrame, name: str = "test") -> bool:
    
    path = Path("data")
    path.mkdir(exist_ok=True)
    
    try:
        data.to_csv(str(path) + f"/{name}.csv", index=True)
        return True
    except Exception as e:
        print("[Erro CSV]: ", e)
        return False

In [4]:
# Obtendo a lista de datasets do OpenML:
all_datasets = openml.datasets.list_datasets(output_format='dataframe')
print(f"Total de datasets no OpenML: {len(all_datasets)}")

# Filtrando os datasets com base nos critérios especificados:

filtered_datasets = all_datasets[
    (all_datasets['NumberOfInstances'] >= 200) &
    (all_datasets['NumberOfInstances'] <= 10000) &
    (all_datasets['NumberOfFeatures'] >= 2) &
    (all_datasets['NumberOfFeatures'] <= 50) &
    (all_datasets['NumberOfClasses'] >= 2) &
    (all_datasets['NumberOfClasses'] <= 6) &
    (all_datasets['NumberOfInstancesWithMissingValues'] < all_datasets['NumberOfInstances'] * 0.3) &
    (all_datasets['MinorityClassSize'] >= 20) &
    (all_datasets['NumberOfNumericFeatures'] == all_datasets['NumberOfFeatures'] - 1) &
    (all_datasets['NumberOfInstances'] * all_datasets['NumberOfFeatures'] <= 60000)
].drop_duplicates(subset='name').reset_index(drop=True)

print(f"Quantidade de datasets após filtro: {len(filtered_datasets)}")

# Removendo datasets da mesma família (baseado no nome):
filtered_datasets['family'] = filtered_datasets['name'].str.extract(r'([a-zA-Z]+)')[0]
final_datasets = filtered_datasets.drop_duplicates(subset='family').reset_index(drop=True) 
# Drop FOREX datasets:
final_datasets = final_datasets[~final_datasets['name'].str.contains('FOREX', case=False)].reset_index(drop=True)
print(f"Quantidade de datasets após remoção de famílias: {len(final_datasets)}")
# print("Datasets selecionados:")
# print(final_datasets[['name', 'NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses']])

Total de datasets no OpenML: 6408
Quantidade de datasets após filtro: 329
Quantidade de datasets após remoção de famílias: 94


In [5]:
# Baixando os datasets filtrados:

datasets = {}   # {nome: {'data': X, 'target': y}}
failed = []

for i, row in final_datasets.iterrows():
    did = int(row['did'])
    name = row['name']

    try:
        ds = openml.datasets.get_dataset(did, download_data=True,
                                         download_qualities=False,
                                         download_features_meta_data=False)
        X, y, _, _ = ds.get_data(dataset_format='dataframe',
                                 target=ds.default_target_attribute)

        unique_name = f"{name}_did{did}" # No caso de haver versões duplicadas

        datasets[unique_name] = {'data': X, 'target': y}
        print(f"[Sucesso]: {i}: {unique_name}  ({X.shape[0]}×{X.shape[1]})")

    except Exception as e:
        failed.append(name)
        print(f"[Erro]: {i}: {name}: {e}")

print("\nDownload concluído!")
print(f"Datasets carregados com sucesso: {len(datasets)}")
print(f"Número de falhas: {len(failed)}")

[Sucesso]: 0: balance-scale_did11  (625×4)
[Sucesso]: 1: breast-w_did15  (699×9)
[Sucesso]: 2: diabetes_did37  (768×8)
[Sucesso]: 3: heart-statlog_did53  (270×13)
[Sucesso]: 4: vehicle_did54  (846×18)
[Sucesso]: 5: ionosphere_did59  (351×34)
[Sucesso]: 6: oil_spill_did311  (937×49)
[Sucesso]: 7: SPECTF_did337  (349×44)
[Sucesso]: 8: prnn_synth_did464  (250×2)
[Sucesso]: 9: rmftsa_sleepdata_did679  (1024×2)
[Sucesso]: 10: fri_c3_1000_25_did715  (1000×25)
[Sucesso]: 11: pwLinear_did721  (200×10)
[Sucesso]: 12: analcatdata_supreme_did728  (4052×7)
[Sucesso]: 13: machine_cpu_did733  (209×6)
[Sucesso]: 14: space_ga_did737  (3107×6)
[Sucesso]: 15: pm10_did750  (500×7)
[Sucesso]: 16: strikes_did770  (625×6)
[Sucesso]: 17: quake_did772  (2178×3)
[Sucesso]: 18: disclosure_x_bias_did774  (662×3)
[Sucesso]: 19: bodyfat_did778  (252×14)
[Sucesso]: 20: delta_ailerons_did803  (7129×5)
[Sucesso]: 21: chscase_vine2_did814  (468×2)
[Sucesso]: 22: chatfield_4_did820  (235×12)
[Sucesso]: 23: stock_did841

In [6]:
# Define classifiers
from sklearn.impute import SimpleImputer

classifiers = {
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'Perceptron': Perceptron(random_state=42, max_iter=1000),
    'MLP': MLPClassifier(random_state=42, max_iter=1000)
}

# Store results
results = []

# Iterate through datasets
for dataset_name, dataset in datasets.items():
    print(f'Processing dataset: {dataset_name}')
    
    # Get data and target
    X = dataset['data']
    y = dataset['target']
    
    # Handle missing values
    imputer = KNNImputer(n_neighbors=3)
    try:
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    except Exception as e:
        print(f'Warning: Imputation failed for {dataset_name} with error: {e}')
        print('Falling back to simple imputation strategy.')
        # Use most frequent strategy for string/categorical data
        imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        # Fit and transform the data
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)


    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Encode target if necessary
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    
    # Evaluate each classifier
    for clf_name, clf in classifiers.items():
        print(f'  Evaluating {clf_name}...', end=' ')
        
        # 5-fold cross validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_results = cross_validate(clf, X, y, cv=cv, scoring='accuracy', 
                                    return_train_score=False)
        
        # Extract fold accuracies
        fold_accs = cv_results['test_score']
        
        # Create result row
        result_row = {
            'Dataset': dataset_name,
            'Classifier': clf_name,
            'acc_fold1': fold_accs[0],
            'acc_fold2': fold_accs[1],
            'acc_fold3': fold_accs[2],
            'acc_fold4': fold_accs[3],
            'acc_fold5': fold_accs[4],
            'acc_mean': fold_accs.mean(),
            'acc_stddev': fold_accs.std(),
            'train_time': cv_results['fit_time'].sum(),
            'test_time': cv_results['score_time'].sum()
        }
        
        results.append(result_row)
        print('Done')

# Create results DataFrame
performances_df = pd.DataFrame(results)

Processing dataset: balance-scale_did11
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: breast-w_did15
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: diabetes_did37
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: heart-statlog_did53
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: vehicle_did54
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evalua

In [8]:
import os
warnings.filterwarnings("ignore")
os.environ['PYTHONWARNINGS'] = 'ignore'

# Extract meta-features
meta_features = []

for dataset_name, dataset in list(datasets.items()):
    print(f'Extracting meta-features from {dataset_name}...', end=' ')
    
    # Get data and target
    X = dataset['data']
    y = dataset['target']

    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Handle missing values
    imputer = KNNImputer(n_neighbors=3)
    try:
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    except Exception as e:
        imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # Encode target if necessary
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    else:
        # Convert to numpy array if it's a pandas Series
        y = np.array(y)
    
    # Extract meta-features
    try:
        mfe = MFE(groups=['landmarking', 'general', 'statistical',
                           'model-based', 'info-theory', 'relative',
                            'clustering', 'complexity', 'itemset', 
                            'concept'],
                           summary=['median', 'min', 'max', 'mean', 
                                    'sd', 'quantiles', 'histogram'])
        # Ignore warnings during meta-feature extraction:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mfe.fit(X.values, y)
            ft = mfe.extract()
        # reset warnings filter to default after extraction:
        warnings.resetwarnings()        
        
        # Create result row with dataset name and meta-features
        result_row = {'dataset': dataset_name}
        
        meta_features.append(pd.DataFrame(dict(zip(ft[0], ft[1])), index=[dataset_name]))
        print('Done')
    except Exception as e:
        print(f'Error: {e}')

# Create meta-features DataFrame
meta_features_df = pd.concat(meta_features, ignore_index=False)

Extracting meta-features from balance-scale_did11... Done
Extracting meta-features from breast-w_did15... Done
Extracting meta-features from diabetes_did37... Done
Extracting meta-features from heart-statlog_did53... Done
Extracting meta-features from vehicle_did54... Done
Extracting meta-features from ionosphere_did59... Done
Extracting meta-features from oil_spill_did311... Done
Extracting meta-features from SPECTF_did337... Done
Extracting meta-features from prnn_synth_did464... Done
Extracting meta-features from rmftsa_sleepdata_did679... Done
Extracting meta-features from fri_c3_1000_25_did715... Done
Extracting meta-features from pwLinear_did721... Done
Extracting meta-features from analcatdata_supreme_did728... Done
Extracting meta-features from machine_cpu_did733... Done
Extracting meta-features from space_ga_did737... Done
Extracting meta-features from pm10_did750... Done
Extracting meta-features from strikes_did770... Done
Extracting meta-features from quake_did772... Done
Ex

In [9]:
performances_df2 = performances_df.pivot(index='Dataset', columns='Classifier', values='acc_mean')
performances_df2.columns.name = None
performances_df2 = performances_df2.reset_index()
performances_df2

,Dataset,DecisionTree,KNN,LogisticRegression,MLP,Perceptron,SVM
0,Apple_Stock_Price_Trends_(2014-2023)_did46420,0.353728,0.350561,0.382351,0.351341,0.351368,0.377186
1,CPMP-2015-runtime-classification_did41919,0.430710,0.497125,0.563630,0.563594,0.493513,0.521905
2,CastMetal1_did1447,0.807459,0.859441,0.856597,0.663497,0.862378,0.871608
3,CostaMadre1_did1446,0.824181,0.858136,0.861469,0.636554,0.280904,0.871638
4,Creditability-German-Credit-Data_did46416,0.685000,0.648000,0.749000,0.700000,0.611000,0.711000
...,...,...,...,...,...,...,...
89,wall-robot-navigation_did1525,1.000000,0.984604,0.921376,0.984604,0.834680,0.958577
90,waveform_did47155,0.717500,0.801250,0.847500,0.787500,0.773750,0.832500
91,wdbc_did1510,0.910402,0.935010,0.950815,0.927993,0.880640,0.913895
92,wilt_did40983,0.976856,0.979748,0.967971,0.973136,0.933660,0.946063


In [10]:
meta_dataset = performances_df2.merge(
    right=meta_features_df, 
    left_on='Dataset', 
    right_index=True, 
    how='left'
)

# Reorder columns: 'Dataset' first, then meta-features, then classifiers
meta_cols = meta_features_df.columns.tolist()
classifier_cols = performances_df2.columns.drop('Dataset').tolist()
meta_dataset = meta_dataset[['Dataset'] + meta_cols + classifier_cols]

In [11]:
data = pd.DataFrame(meta_dataset)

if save_csv(data=data, name="meta_dataset"):
    print("OK! Csv criado com sucesso!")
else:
    print("Erro ao tentar salvar csv!")

OK! Csv criado com sucesso!
